In [3]:
# =====================================
# 1. IMPORT LIBRARIES
# =====================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample
import joblib

# =====================================
# 2. LOAD DATA
# =====================================
df = pd.read_csv("preprocessing_dataset.csv")

print("Original Shape:", df.shape)
print(df.dtypes)

# =====================================
# 3. HANDLE CLASS IMBALANCE (Oversampling)
# =====================================
majority = df[df.Purchased == 0]
minority = df[df.Purchased == 1]

minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

df_balanced = pd.concat([majority, minority_upsampled])

print("Balanced Shape:", df_balanced.shape)
print(df_balanced["Purchased"].value_counts())

# =====================================
# 4. FEATURES & TARGET
# =====================================
X = df_balanced.drop("Purchased", axis=1)
y = df_balanced["Purchased"]

# =====================================
# 5. COLUMN TYPES
# =====================================
categorical_cols = ["Name", "City", "Gender"]
numeric_cols = ["Age", "Salary"]

# =====================================
# 6. PREPROCESSING PIPELINE
# =====================================
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ]
)

# =====================================
# 7. FULL PIPELINE (High Accuracy Model)
# =====================================
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

# =====================================
# 8. TRAIN TEST SPLIT
# =====================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

# =====================================
# 9. TRAIN MODEL
# =====================================
pipeline.fit(X_train, y_train)

# =====================================
# 10. PREDICTIONS
# =====================================
predictions = pipeline.predict(X_test)

# =====================================
# 11. EVALUATION
# =====================================
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

print("\nClassification Report:\n", classification_report(y_test, predictions))

# =====================================
# 12. SAVE PIPELINE
# =====================================
joblib.dump(pipeline, "final_pipeline.pkl")

print("Pipeline Saved!")

# =====================================
# 13. LOAD PIPELINE
# =====================================
loaded_pipeline = joblib.load("final_pipeline.pkl")

# =====================================
# 14. TEST ON NEW DATA
# =====================================
new_data = pd.DataFrame({
    "Name": ["Ali", "Sara"],
    "City": ["Lahore", "Karachi"],
    "Gender": ["Male", "Female"],
    "Age": [25, 30],
    "Salary": [50000, 80000]
})

new_predictions = loaded_pipeline.predict(new_data)

print("\nNew Predictions:", new_predictions)

Original Shape: (500, 6)
Name         object
City         object
Gender       object
Age           int64
Salary        int64
Purchased     int64
dtype: object
Balanced Shape: (726, 6)
Purchased
0    363
1    363
Name: count, dtype: int64
Train Shape: (580, 5)
Test Shape: (146, 5)
Accuracy: 0.821917808219178

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.72      0.82        81
           1       0.73      0.95      0.83        65

    accuracy                           0.82       146
   macro avg       0.84      0.83      0.82       146
weighted avg       0.85      0.82      0.82       146

Pipeline Saved!

New Predictions: [1 0]
